In [2]:
import pandas as pd
import numpy as np
import sqlite3

df = pd.read_csv('../data/raw/superstore.csv', encoding='latin-1')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))

Shape: (9994, 21)
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
   Row ID        Order ID Order Date   Ship Date     Ship Mode Customer ID  \
0       1  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   
1       2  CA-2016-152156  11/8/2016  11/11/2016  Second Class    CG-12520   
2       3  CA-2016-138688  6/12/2016   6/16/2016  Second Class    DV-13045   

     Customer Name    Segment        Country         City  ... Postal Code  \
0      Claire Gute   Consumer  United States    Henderson  ...       42420   
1      Claire Gute   Consumer  United States    Henderson  ...       42420   
2  Darrin Van Huff  Corporate  United States  Los Angeles  ...       90036   

   Region       Product ID         Category Sub-Category  \
0   South  FUR-BO-10001798 

In [3]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Fix date columns so Python understands them as dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Clean up text — remove extra spaces, make consistent
df['Category'] = df['Category'].str.strip().str.title()
df['Region'] = df['Region'].str.strip().str.title()
df['Segment'] = df['Segment'].str.strip().str.title()

# Rename all columns — replace spaces with underscores
# So instead of "Order Date" it becomes "Order_Date"
# This makes SQL queries easier later
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

print("Any empty values?", df.isnull().sum().sum())
print("Clean shape:", df.shape)

Any empty values? 0
Clean shape: (9994, 21)


In [6]:
import os
os.makedirs('../data/processed', exist_ok=True)
print("Folder ready ✅")

Folder ready ✅


In [4]:
# Pull out year, month, quarter from the order date
df['Order_Year'] = df['Order_Date'].dt.year
df['Order_Month'] = df['Order_Date'].dt.month
df['Order_Month_Name'] = df['Order_Date'].dt.strftime('%b')  # Jan, Feb etc
df['Order_Quarter'] = df['Order_Date'].dt.quarter

# How many days did shipping take?
df['Shipping_Delay_Days'] = (df['Ship_Date'] - df['Order_Date']).dt.days

# Profit margin = what % of the sale price was profit
df['Profit_Margin_Pct'] = (df['Profit'] / df['Sales'] * 100).round(2)

# Was this order a loss? 1 = yes, 0 = no
df['Is_Loss'] = (df['Profit'] < 0).astype(int)

# Group orders into revenue buckets
df['Revenue_Band'] = pd.cut(
    df['Sales'],
    bins=[0, 100, 500, 1000, 99999],
    labels=['Low', 'Mid', 'High', 'Premium']
)

# Preview the new columns
print(df[['Sales', 'Profit', 'Profit_Margin_Pct', 
          'Shipping_Delay_Days', 'Is_Loss', 'Revenue_Band']].head(10))

      Sales    Profit  Profit_Margin_Pct  Shipping_Delay_Days  Is_Loss  \
0  261.9600   41.9136              16.00                    3        0   
1  731.9400  219.5820              30.00                    3        0   
2   14.6200    6.8714              47.00                    4        0   
3  957.5775 -383.0310             -40.00                    7        1   
4   22.3680    2.5164              11.25                    7        0   
5   48.8600   14.1694              29.00                    5        0   
6    7.2800    1.9656              27.00                    5        0   
7  907.1520   90.7152              10.00                    5        0   
8   18.5040    5.7825              31.25                    5        0   
9  114.9000   34.4700              30.00                    5        0   

  Revenue_Band  
0          Mid  
1         High  
2          Low  
3         High  
4          Low  
5          Low  
6          Low  
7         High  
8          Low  
9          Mid 

In [7]:
# Create a database file and save everything into it
conn = sqlite3.connect('../data/processed/superstore.db')

df.to_sql('orders', conn, if_exists='replace', index=False)

# Quick check — count rows in the database
result = pd.read_sql("SELECT COUNT(*) as total_rows FROM orders", conn)
print("Rows saved to database:", result)

conn.close()
print("✅ ETL complete! Database saved to data/processed/superstore.db")

Rows saved to database:    total_rows
0        9994
✅ ETL complete! Database saved to data/processed/superstore.db


In [8]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/processed/superstore.db')

# Pull the business_insights view into a dataframe
df_insights = pd.read_sql("SELECT * FROM business_insights", conn)

# Save it as CSV
df_insights.to_csv('../data/processed/business_insights.csv', index=False)

conn.close()
print("✅ Done! Check data/processed/ folder")
print(df_insights.head())

✅ Done! Check data/processed/ folder
    Region         Category   Revenue   Profit  Margin_Pct  Avg_Discount_Pct  \
0  Central        Furniture  163797.0  -2871.0       -18.6              29.7   
1  Central  Office Supplies  167026.0   8880.0       -15.9              25.3   
2     East        Furniture  208291.0   3046.0         9.2              15.4   
3     West        Furniture  252613.0  11505.0         9.8              13.1   
4     East       Technology  264974.0  47462.0        12.6              14.3   

   Loss_Orders                                    Business_Signal  
0          317  CRITICAL: Negative margin — review pricing imm...  
1          375  CRITICAL: Negative margin — review pricing imm...  
2          184       HEALTHY: Performance within acceptable range  
3          155       HEALTHY: Performance within acceptable range  
4          136       HEALTHY: Performance within acceptable range  
